In [43]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, make_scorer
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from collections import Counter

In [44]:
df=pd.read_csv("../Datasets/Travel personality data - Form Responses.csv")

In [45]:
df.head()

,Timestamp,Enter your name:,Select your age group :,Gender:,"1. When traveling, I value excitement (thrill)",2. I like places that offer historical attractions / monuments,3. I avoid typical tourist places whenever possible.,"4. I like physically demanding travel experiences (trekking, rafting, long walks).",5. I usually make many new friends during my travels,6. I prefer places that teach me something and broaden my knowledge than simply help me get mind off work and everyday life.,...,"9. I prefer calm, quiet destinations over crowded ones.",10. I prefer destinations that push me out of my comfort zone.,11. I prefer destinations known for culture and heritage over purely scenic locations.,12. I travel mainly to relax my mind rather than seek excitement.,Nature-Based Destinations :,Historical & Cultural Places :,Religious / Spiritual Destinations :,Beach & Adventure Destinations :,Urban / City-Centric Destinations :,How would you describe yourself as a traveler?
0,2/1/2026 9:27:36,Mukul Aggarwal,18 to 30,Male,3,5,4,1,2,4,...,5,3,4,4,4,4,4,3,3,Social Butterfly (The Cultural Explorer)
1,2/1/2026 13:55:58,Lavanya Agrawal,18 to 30,Female,3,5,4,1,2,4,...,5,3,4,4,4,4,4,3,3,Social Butterfly (The Cultural Explorer)
2,2/1/2026 11:33:31,Arnav,18 to 30,Male,4,2,4,1,4,4,...,5,2,4,5,5,2,3,4,1,Peace-Seeking Traveler (The Quiet Voyager)
3,2/1/2026 11:33:48,Jahnavi Shringi,18 to 30,Female,4,2,3,3,2,4,...,2,3,3,4,3,2,4,4,3,Social Butterfly (The Cultural Explorer)
4,2/1/2026 11:33:49,Pushkar Agrawal,18 to 30,Male,4,4,3,2,4,3,...,5,3,3,4,4,4,3,4,4,Peace-Seeking Traveler (The Quiet Voyager)


In [46]:
df=df.drop(columns=["Timestamp", "Enter your name:", "Select your age group :", "Gender:"])

In [47]:
df = df.rename(columns={
    "1. When traveling, I value excitement (thrill)": "Q1",
    "2. I like places that offer historical attractions / monuments ": "Q2",
    "3. I avoid typical tourist places whenever possible.": "Q3",
    "4. I like physically demanding travel experiences (trekking, rafting, long walks).": "Q4",
    "5. I usually make many new friends during my travels": "Q5",
    "6. I prefer places that teach me something and broaden my knowledge than simply help me get mind off work and everyday life.": "Q6",
    "7. When traveling, I prefer to spend time with other people than alone.": "Q7",
    "8. I like visiting busy markets, festivals, and cultural events.": "Q8",
    "9. I prefer calm, quiet destinations over crowded ones.": "Q9",
    "10. I prefer destinations that push me out of my comfort zone.": "Q10",
    "11. I prefer destinations known for culture and heritage over purely scenic locations.": "Q11",
    "12. I travel mainly to relax my mind rather than seek excitement.": "Q12",
    "Nature-Based Destinations :" : "Nature Based",
    "Historical & Cultural Places :": "Historical & Cultural",
    "Religious / Spiritual Destinations :" : "Religious / Spiritual",
    "Beach & Adventure Destinations :":"Beach & Adventure",
    "Urban / City-Centric Destinations :": "Urban / City-Centric",
    "How would you describe yourself as a traveler?": "Travel Personality"
})

In [48]:
df.head()

,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Nature Based,Historical & Cultural,Religious / Spiritual,Beach & Adventure,Urban / City-Centric,Travel Personality
0,3,5,4,1,2,4,4,3,5,3,4,4,4,4,4,3,3,Social Butterfly (The Cultural Explorer)
1,3,5,4,1,2,4,4,3,5,3,4,4,4,4,4,3,3,Social Butterfly (The Cultural Explorer)
2,4,2,4,1,4,4,1,1,5,2,4,5,5,2,3,4,1,Peace-Seeking Traveler (The Quiet Voyager)
3,4,2,3,3,2,4,4,5,2,3,3,4,3,2,4,4,3,Social Butterfly (The Cultural Explorer)
4,4,4,3,2,4,3,4,4,5,3,3,4,4,4,3,4,4,Peace-Seeking Traveler (The Quiet Voyager)


In [49]:
le = LabelEncoder()
df['Travel Personality'] = le.fit_transform(df['Travel Personality'])

In [50]:
X = df[[f"Q{i}" for i in range(1, 13)]].values
y = df['Travel Personality'].values

In [51]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## Random forest with smote, not tuned

In [52]:
smote = SMOTE(k_neighbors=6, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

rf = RandomForestClassifier(random_state=42)

rf.fit(X_train_sm, y_train_sm)

y_pred = rf.predict(X_test)

print("Random Forest with SMOTE (no tuning)")
print(classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='macro'))

Random Forest with SMOTE (no tuning)
              precision    recall  f1-score   support

           0       0.60      0.38      0.46         8
           1       0.60      0.82      0.69        11
           2       0.60      0.50      0.55         6

    accuracy                           0.60        25
   macro avg       0.60      0.56      0.57        25
weighted avg       0.60      0.60      0.58        25

F1 Score: 0.5664335664335663


## Random Forest (NO SMOTE, tuned)

In [53]:
rf_no_smote = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42
)

rf_no_smote.fit(X_train, y_train)

y_pred = rf_no_smote.predict(X_test)

print("RF without SMOTE")
print(classification_report(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred, average='macro'))

RF without SMOTE
              precision    recall  f1-score   support

           0       0.33      0.25      0.29         8
           1       0.73      0.73      0.73        11
           2       0.50      0.67      0.57         6

    accuracy                           0.56        25
   macro avg       0.52      0.55      0.53        25
weighted avg       0.55      0.56      0.55        25

F1: 0.5281385281385281


## Random Forest (WITH SMOTE, tuned)

In [54]:
smote = SMOTE(k_neighbors=2, random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

rf_smote = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)

rf_smote.fit(X_train_sm, y_train_sm)

y_pred = rf_smote.predict(X_test)

print("RF with SMOTE")
print(classification_report(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred, average='macro'))

RF with SMOTE
              precision    recall  f1-score   support

           0       0.40      0.25      0.31         8
           1       0.69      0.82      0.75        11
           2       0.57      0.67      0.62         6

    accuracy                           0.60        25
   macro avg       0.55      0.58      0.56        25
weighted avg       0.57      0.60      0.58        25

F1: 0.5576923076923077


## SVM (WITH SMOTE)

In [55]:
smote = SMOTE(k_neighbors=2, random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

svm_smote = SVC(
    kernel='rbf',
    C=1,
    gamma='scale',
    class_weight='balanced'
)

svm_smote.fit(X_train_sm, y_train_sm)

y_pred = svm_smote.predict(X_test)

print("SVM with SMOTE")
print(classification_report(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred, average='macro'))

SVM with SMOTE
              precision    recall  f1-score   support

           0       0.44      0.50      0.47         8
           1       0.70      0.64      0.67        11
           2       0.67      0.67      0.67         6

    accuracy                           0.60        25
   macro avg       0.60      0.60      0.60        25
weighted avg       0.61      0.60      0.60        25

F1: 0.6013071895424836


## SVM (NO SMOTE)

In [56]:
svm_no_smote = SVC(
    kernel='rbf',
    C=1,
    gamma='scale',
    class_weight='balanced'
)

svm_no_smote.fit(X_train, y_train)

y_pred = svm_no_smote.predict(X_test)

print("SVM without SMOTE")
print(classification_report(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred, average='macro'))

SVM without SMOTE
              precision    recall  f1-score   support

           0       0.50      0.62      0.56         8
           1       0.75      0.55      0.63        11
           2       0.57      0.67      0.62         6

    accuracy                           0.60        25
   macro avg       0.61      0.61      0.60        25
weighted avg       0.63      0.60      0.60        25

F1: 0.600839706102864


## SVM with k fold cross val, using smote

In [57]:
pipeline = Pipeline([
    ('smote', SMOTE(k_neighbors=2, random_state=42)),
    ('svm', SVC(kernel='rbf', C=1, gamma='scale', class_weight='balanced'))
])

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

f1_macro = make_scorer(f1_score, average='macro')

scores = cross_val_score(pipeline, X, y, cv=skf, scoring=f1_macro)

print("F1 scores for each fold:", scores)
print("Mean F1 score:", scores.mean())

F1 scores for each fold: [0.42430149 0.56188424 0.64886965]
Mean F1 score: 0.5450184599332614


## Random Forest implementation (from scratch)

In [58]:
class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

    def fit(self, X, y):
        self.n_classes = len(set(y))
        self.tree = self._grow_tree(X, y)

    def _gini(self, y):
        counts = np.bincount(y)
        probs = counts / len(y)
        return 1 - np.sum(probs ** 2)

    def _best_split(self, X, y):
        best_feature, best_thresh, best_gain = None, None, -1
        parent_gini = self._gini(y)

        n_features = X.shape[1]

        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])
            for t in thresholds:
                left_idx = X[:, feature] <= t
                right_idx = X[:, feature] > t

                if sum(left_idx) == 0 or sum(right_idx) == 0:
                    continue

                gini_left = self._gini(y[left_idx])
                gini_right = self._gini(y[right_idx])

                weighted_gini = (
                    (len(y[left_idx]) * gini_left +
                     len(y[right_idx]) * gini_right) / len(y)
                )

                gain = parent_gini - weighted_gini

                if gain > best_gain:
                    best_feature = feature
                    best_thresh = t
                    best_gain = gain

        return best_feature, best_thresh

    def _grow_tree(self, X, y, depth=0):
        if (depth >= self.max_depth or
            len(set(y)) == 1 or
            len(y) < self.min_samples_split):
            return Counter(y).most_common(1)[0][0]

        feature, thresh = self._best_split(X, y)

        if feature is None:
            return Counter(y).most_common(1)[0][0]

        left_idx = X[:, feature] <= thresh
        right_idx = X[:, feature] > thresh

        left = self._grow_tree(X[left_idx], y[left_idx], depth + 1)
        right = self._grow_tree(X[right_idx], y[right_idx], depth + 1)

        return (feature, thresh, left, right)

    def _predict_one(self, x, node):
        if not isinstance(node, tuple):
            return node

        feature, thresh, left, right = node

        if x[feature] <= thresh:
            return self._predict_one(x, left)
        else:
            return self._predict_one(x, right)

    def predict(self, X):
        return np.array([self._predict_one(x, self.tree) for x in X])

In [59]:
class RandomForest:
    def __init__(self, n_trees=10, max_depth=5):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []

    def _bootstrap(self, X, y):
        n_samples = X.shape[0]
        idx = np.random.choice(n_samples, n_samples, replace=True)
        return X[idx], y[idx]

    def fit(self, X, y):
        self.trees = []

        for _ in range(self.n_trees):
            tree = DecisionTree(max_depth=self.max_depth)
            X_sample, y_sample = self._bootstrap(X, y)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        tree_preds = np.swapaxes(tree_preds, 0, 1)

        final_preds = []
        for preds in tree_preds:
            final_preds.append(Counter(preds).most_common(1)[0][0])

        return np.array(final_preds)

In [60]:
rf = RandomForest(n_trees=10, max_depth=5)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

f1 = f1_score(y_test, y_pred, average='macro')
print("F1 Score:", f1)

Classification Report:

              precision    recall  f1-score   support

           0       0.17      0.12      0.14         8
           1       0.50      0.64      0.56        11
           2       0.60      0.50      0.55         6

    accuracy                           0.44        25
   macro avg       0.42      0.42      0.42        25
weighted avg       0.42      0.44      0.42        25

F1 Score: 0.4161038961038961
